# Lab 5, Day 1 — Exploration and Cleaning

Explore, profile, and clean the Titanic dataset, then split it and check your column
groups - no model yet. See `Lab5_Day1_Instructions.md` for the full walkthrough.

This notebook is just a shell: it gives you a place to write and document your work,
but the profiling, the decisions, and the reasoning are yours.

In [1]:
import pandas as pd
import numpy as np
from pipeline import split_data, get_column_groups

from data import load_titanic

df, source = load_titanic()
print('source:', source)
df.head()


Loaded real Titanic from OpenML  (1309, 14)
source: openml


,Pclass,Survived,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


## Step 1: Load and profile

In [2]:
# TODO: df.info(), df.describe(include='all').T, and missingness percentage per
# column, sorted descending
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   Pclass     1309 non-null   int64   
 1   Survived   1309 non-null   int64   
 2   Name       1309 non-null   object  
 3   Sex        1309 non-null   category
 4   Age        1046 non-null   float64 
 5   SibSp      1309 non-null   int64   
 6   Parch      1309 non-null   int64   
 7   Ticket     1309 non-null   object  
 8   Fare       1308 non-null   float64 
 9   Cabin      295 non-null    object  
 10  Embarked   1307 non-null   category
 11  boat       486 non-null    object  
 12  body       121 non-null    float64 
 13  home.dest  745 non-null    object  
dtypes: category(2), float64(3), int64(4), object(5)
memory usage: 125.7+ KB


In [3]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Pclass,1309.0,NaN,NaN,NaN,2.294882,0.837836,1.0,2.0,3.0,3.0,3.0
Survived,1309.0,NaN,NaN,NaN,0.381971,0.486055,0.0,0.0,0.0,1.0,1.0
Name,1309,1307,"Connolly, Miss. Kate",2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sex,1309,2,male,843,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,1046.0,NaN,NaN,NaN,29.881135,14.4135,0.1667,21.0,28.0,39.0,80.0
SibSp,1309.0,NaN,NaN,NaN,0.498854,1.041658,0.0,0.0,0.0,1.0,8.0
Parch,1309.0,NaN,NaN,NaN,0.385027,0.86556,0.0,0.0,0.0,0.0,9.0
Ticket,1309,929,CA. 2343,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Fare,1308.0,NaN,NaN,NaN,33.295479,51.758668,0.0,7.8958,14.4542,31.275,512.3292
Cabin,295,186,C23 C25 C27,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
(df.isna().mean() * 100).sort_values(ascending=False)

body         90.756303
Cabin        77.463713
boat         62.872422
home.dest    43.086325
Age          20.091673
Embarked      0.152788
Fare          0.076394
SibSp         0.000000
Name          0.000000
Survived      0.000000
Pclass        0.000000
Sex           0.000000
Parch         0.000000
Ticket        0.000000
dtype: float64

## Step 2: The most skipped checks - does missingness predict the target?

In [5]:
# TODO: for at least Age and Cabin, compare Survived rates between rows where the
# column is missing vs not. Is there a difference worth caring about?
a = sum(df["Age"].isna())

c = sum(df["Cabin"].isna())

print(f"Age not documentated for {a} passengers")
print(f"Cabin Number not documented for {c} passengers")

Age not documentated for 263 passengers
Cabin Number not documented for 1014 passengers


In [6]:
df.groupby(df['Cabin'].isna())['Survived'].agg(
        count='count',
        survival_rate='mean'
    ).rename(index={True: 'Missing (NaN)', False: 'Recorded (Present)'})

,count,survival_rate
Cabin,,
Recorded (Present),295,0.654237
Missing (NaN),1014,0.302761


In [7]:
df.groupby(df['Age'].isna())['Survived'].agg(
        count='count',
        survival_rate='mean'
    ).rename(index={True: 'Missing (NaN)', False: 'Recorded (Present)'})

,count,survival_rate
Age,,
Recorded (Present),1046,0.408222
Missing (NaN),263,0.277567


## Step 3: Look for impossible values and placeholders

In [8]:
# TODO: check unique values in object columns, zero/negative fares, absurd ages -\n
# anything that looks like a placeholder rather than real missingness\n
for col in df.select_dtypes(include="object").columns:
    print(f"\n{col}:")
    print(df[col].unique())


Name:
['Allen, Miss. Elisabeth Walton' 'Allison, Master. Hudson Trevor'
 'Allison, Miss. Helen Loraine' ... 'Zakarian, Mr. Mapriededer'
 'Zakarian, Mr. Ortin' 'Zimmerman, Mr. Leo']

Ticket:
['24160' '113781' '19952' '13502' '112050' '11769' 'PC 17609' 'PC 17757'
 'PC 17477' '19877' '27042' 'PC 17318' 'PC 17558' '11813' '13050' '11751'
 '111369' 'PC 17483' '13905' '11967' 'PC 17760' '110564' '113784' '112277'
 '36928' '113783' '110489' 'PC 17608' '113505' '111427' '113054'
 'PC 17591' '112379' 'PC 17610' '16966' '113050' '113798' 'PC 17476'
 'PC 17606' 'PC 17755' '695' '113059' '113760' '19924' '17770'
 'W.E.P. 5734' '113806' '110152' 'PC 17594' '112051' '13508' '110465'
 '5727' 'PC 17756' '11770' '113791' 'WE/P 5735' '112901' 'PC 17599'
 '113055' '113804' 'F.C. 12750' '17474' '33638' 'PC 17761' '11755'
 'PC 17485' 'PC 17580' '11767' '36947' 'PC 17531' 'PC 17598' '17421'
 'PC 17474' '113051' '19950' '113778' 'PC 17611' '17765' '13568' '13567'
 '112058' '113803' '111320' '113503' '11237

In [9]:
bad_fares = (df["Fare"] <= 0).sum()

print(f"Bad Fares: {bad_fares}")
df[df["Fare"] <= 0].head(5)

Bad Fares: 17


,Pclass,Survived,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,boat,body,home.dest
7,1,0,"Andrews, Mr. Thomas Jr",male,39.0,0,0,112050,0.0,A36,S,NaN,NaN,"Belfast, NI"
70,1,0,"Chisholm, Mr. Roderick Robert Crispin",male,NaN,0,0,112051,0.0,NaN,S,NaN,NaN,"Liverpool, England / Belfast"
125,1,0,"Fry, Mr. Richard",male,NaN,0,0,112058,0.0,B102,S,NaN,NaN,NaN
150,1,0,"Harrison, Mr. William",male,40.0,0,0,112059,0.0,B94,S,NaN,110.0,NaN
170,1,1,"Ismay, Mr. Joseph Bruce",male,49.0,0,0,112058,0.0,B52 B54 B56,S,C,NaN,Liverpool


In [11]:
df["Fare"].describe()

count    1308.000000
mean       33.295479
std        51.758668
min         0.000000
25%         7.895800
50%        14.454200
75%        31.275000
max       512.329200
Name: Fare, dtype: float64

In [12]:
df[df["Age"] <= 0]

,Pclass,Survived,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,boat,body,home.dest


In [13]:
df["Age"].describe()

count    1046.000000
mean       29.881135
std        14.413500
min         0.166700
25%        21.000000
50%        28.000000
75%        39.000000
max        80.000000
Name: Age, dtype: float64

In [14]:
non_numbers = df[pd.to_numeric(df["Fare"], errors='coerce').isna() & df["Fare"].notna()]
print(non_numbers["Fare"])

Series([], Name: Fare, dtype: float64)


In [15]:
placeholders = ["?", "unknown", "na", "n/a", "none", ""]

for col in df.columns:
    bad = df[col].astype(str).str.lower().isin(placeholders)

    if bad.any():
        print(f"\n--- {col} ---")
        print(df.loc[bad, col].value_counts())

In [16]:
print("Target column:", "Survived")

for col in df.columns:
    if col != "Survived":
        print(f"{col}: {df[col].nunique()} unique values")

Target column: Survived
Pclass: 3 unique values
Name: 1307 unique values
Sex: 2 unique values
Age: 98 unique values
SibSp: 7 unique values
Parch: 8 unique values
Ticket: 929 unique values
Fare: 281 unique values
Cabin: 186 unique values
Embarked: 3 unique values
boat: 27 unique values
body: 121 unique values
home.dest: 369 unique values


## Step 4: Decide, and write it down

For each column with a problem, add a markdown cell (or a row in a table here) saying
what you did and why. Code with no commentary earns much less credit than the same
code with one sentence of justification.

### Cleaning Decisions

#### Age

* **Problem found:** About 20% of the `Age` values are missing (263 rows).
* **What we found:** The survival rate is **40.8%** when `Age` is present, compared with **27.8%** when `Age` is missing.
* **Decision:** Keep the `Age` column. In Day 2, handle the missing values using imputation, preferably using the median based on relevant groups such as title/class. Also create an `age_missing` indicator.
* **Reason:** The difference in survival rates shows that the missingness itself may contain useful information. Dropping the rows or the entire column would lose data and potentially remove this signal.

#### Cabin

* **Problem found:** About 77% of the `Cabin` values are missing (1014 rows).
* **What we found:** The survival rate is **65.4%** when `Cabin` is present, compared with **30.3%** when it is missing.
* **Decision:** Do not use the raw `Cabin` string directly. In Day 2, create a binary `has_cabin` feature indicating whether a cabin was recorded, and then drop the raw `Cabin` column.
* **Reason:** `Cabin` has many unique values and is very sparse, so directly encoding the cabin strings would create a high-cardinality feature. However, whether a cabin was recorded is strongly associated with survival, so keeping this information as a binary indicator preserves useful signal without unnecessary complexity.

#### Embarked

* **Problem found:** Only **2 values** are missing, which is approximately **0.15%** of the dataset.
* **Decision:** Keep the `Embarked` column and impute the missing values with the mode (`S`) in Day 2.
* **Reason:** The amount of missing data is extremely small. Removing those rows would be unnecessary, so a simple mode imputation is appropriate.

#### Fare

* **Problem found:** There is **1 genuine NaN value** and **17 values that are less than or equal to zero**. All 17 of those values are exactly `0.0`; there are no negative fares.
* **Decision:** Treat the zero fares as invalid values and convert them to `NaN` in Day 2. Then impute the missing `Fare` values, preferably using the median within `Pclass` if appropriate.
* **Reason:** A fare of `0.0` is not a meaningful passenger fare value for this dataset and should not be treated as a normal numerical observation. Converting these values to missing values allows them to be handled consistently during preprocessing.

#### Name

* **Problem found:** `Name` is free text with **1307 unique values**, so it is not suitable as a raw categorical feature.
* **Decision:** Keep `Name` temporarily for Day 2 so that useful information such as titles (`Mr`, `Mrs`, `Miss`, `Master`, etc.) can be extracted. After extracting the title, the full name can be excluded from the modelling features.
* **Reason:** The full name is essentially an identifier and has very high cardinality. However, the title contained within the name can provide useful information about sex, age, and social class.

#### Ticket

* **Problem found:** `Ticket` contains **929 unique values** and consists of mixed alphanumeric ticket identifiers.
* **Decision:** Likely drop `Ticket` in Day 2 unless further exploration shows that extracting a useful ticket-prefix or group-size feature is worthwhile.
* **Reason:** Most ticket values behave like identifiers rather than meaningful categorical values. Its high cardinality makes direct encoding inefficient and unlikely to generalize well.

#### home.dest

* **Problem found:** About **43%** of `home.dest` values are missing, and the column contains around **369 unique values** of free text.
* **Decision:** Drop `home.dest` in Day 2.
* **Reason:** The column has substantial missingness and high-cardinality free-text values. Encoding it directly would add considerable complexity and may introduce noise. Some of the useful geographical information is also partially represented by `Embarked`.

#### Placeholder Values

* **Check performed:** String/object columns were checked for common placeholder values such as `?`, `unknown`, `na`, `n/a`, `none`, and empty strings.
* **Result:** No such placeholder values were found.
* **Decision:** No additional cleaning is required for placeholder strings.
* **Reason:** The missing values identified in the dataset are genuine `NaN` values rather than common placeholder strings.

### Target Leakage

Two columns were identified as containing information that would not be available when making a prediction before the outcome.

#### boat

* `boat` records the lifeboat that a passenger boarded.
* This information is only available after the disaster and is strongly tied to whether the passenger survived.
* **Decision:** Exclude `boat` from the features.

#### body

* `body` contains the body-recovery number for passengers who did not survive.
* This directly reveals information about the outcome.
* **Decision:** Exclude `body` from the features.

Therefore, the target and leakage columns are removed before splitting:

```python
X = df.drop(columns=["Survived", "boat", "body"])
y = df["Survived"]
```

This ensures that `Survived` is the target and that `boat` and `body` cannot leak information about the target into the model.

### Overall Cleaning Plan

At this stage, no model is being built and no preprocessing is being fitted. The decisions above are only being documented for Day 2.

The main plan is:

* Keep `Age` and impute its missing values later, while also preserving an `age_missing` indicator.
* Convert the useful information from `Cabin` into a `has_cabin` indicator and drop the raw cabin string.
* Impute the two missing `Embarked` values with the mode.
* Treat the 17 zero `Fare` values as invalid/missing and impute them together with the existing missing `Fare`.
* Extract useful title information from `Name` before dropping the full name.
* Drop `Ticket` unless further EDA identifies a useful derived feature.
* Drop `home.dest` because of its high missingness and high-cardinality free-text nature.
* No action is required for placeholder strings because none were found.
* Exclude `boat` and `body` because they are target-leakage columns.


## Step 5: Split first, before fitting anything (`pipeline.py`)

In [17]:
# TODO: build X (drop the target and any obvious identifiers/free text you won't
# use directly) and y, then train_test_split with stratify=y and a fixed random_state.
# Nothing should be .fit() on the full dataset before this split exists.
y = df["Survived"]

X = df.drop(columns=["Survived", "boat", "body"])

In [18]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1309, 11)
y shape: (1309,)


In [19]:
X_train, X_test, y_train, y_test = split_data(X, y)

## Step 6: Column groups - and check them by eye

In [20]:
# Step 6: Column groups — check them by eye.
# Pclass must be in categorical (it is ordinal, not a real number).
# Sex and Embarked must be in categorical (they have dtype 'category' from OpenML).
print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)


X_train: (1047, 11)
X_test:  (262, 11)
y_train: (1047,)
y_test:  (262,)


In [21]:
print("Overall survival rate:", y.mean())
print("Training survival rate:", y_train.mean())
print("Testing survival rate:", y_test.mean())

Overall survival rate: 0.3819709702062643
Training survival rate: 0.38204393505253104
Testing survival rate: 0.3816793893129771


In [22]:
numeric_cols, categorical_cols = get_column_groups(X_train)

print("Numeric columns:")
print(numeric_cols)

print("\nCategorical columns:")
print(categorical_cols)

all_grouped = set(numeric_cols) | set(categorical_cols)
missing_from_groups = set(X_train.columns) - all_grouped

print("Columns NOT in any group:", missing_from_groups)


Numeric columns:
['Age', 'SibSp', 'Parch', 'Fare']

Categorical columns:
['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked', 'home.dest', 'Pclass']
Columns NOT in any group: set()


In [23]:
import joblib

joblib.dump(
    {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    },
    "split.joblib"
)

['split.joblib']

In [24]:
import os

print("split.joblib saved:", os.path.exists("split.joblib"))

split.joblib saved: True


---
**Before you close this notebook today:**
- Save your split so Day 2 resumes rather than re-derives it - e.g.
  `joblib.dump({"X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test}, "split.joblib")`.
  Re-splitting tomorrow with a different `random_state` would silently invalidate every
  comparison you make against today's work.
- Confirm your split used `stratify=y` and a fixed `random_state`.
- Make sure Step 4's cleaning decisions are actually written down while the reasoning
  is fresh - reconstructing it tomorrow produces visibly thinner justifications.
- Keep this notebook and folder as-is. Day 2 is a new notebook here, not a restart.